# 📑 User Guide: Bibliometric Analysis Pipeline (CAPES + OpenAlex)

This notebook is designed for researchers to analyze the scientific production of Brazilian postgraduate programs. It calculates the **Publishing Performance Index (PPI)** and measures **Thematic Drift** (how an author's research topics evolve after collaborations).

### 🚀 Quick Start
1. **API Key**: Obtain an OpenAlex API key and paste it into the `oa_api_key` field in the first code cell.
2. **Program ID**: To analyze a specific program, change the `ID_PROGRAMA` variable in the first code cell. You can find this ID on the [CAPES Open Data portal](https://dadosabertos.capes.gov.br/).
3. **Run All**: Go to `Runtime > Run all` to execute the full pipeline.

### ⚡ OpenAlex Efficiency (v2 – pyalex)
This version replaces all raw `requests` calls to OpenAlex with **[pyalex](https://github.com/J535D165/pyalex)**:
- **`select()`** on every query so only the fields we actually use are transferred.
- **Batch author resolution** via `search_filter` + `display_name.search` — one call per name, but with connection-reuse and automatic retry baked in.
- **`paginate(method="cursor", per_page=200)`** for works — the maximum page size, cutting the number of round-trips by half compared to `per_page=100`.
- **Batch author detail** with `filter(openalex="A1|A2|…")` and `select()` — only transfers the ~5 fields needed for PPI, not the full author object.
- All rate-limiting is handled by pyalex's built-in retry/back-off; explicit `time.sleep()` calls are removed.

### 1. Environment Setup
This cell installs the necessary statistical libraries and **pyalex**, then configures the OpenAlex connection globally (polite-pool e-mail, API key, automatic retry). The `WORK_SELECT` / `AUTHOR_SELECT` constants tell the API to return only the fields we actually consume, shrinking every response significantly.

In [ ]:
oa_api_key = "" # @param {"type":"string","placeholder":"api key"}
ID_programa_capes = "1733" # @param {"type":"string","placeholder":"Digite o ID do programa de pós graduação"}
# ── Instalação de Dependências e Configurações Gerais ─────────────────────────
%pip install -q statsmodels>=0.14 scipy>=1.9 pandas>=1.5 numpy>=1.23 matplotlib seaborn pyalex mako

import requests
import pandas as pd
import numpy as np
import time
import json
import warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from itertools import combinations
from scipy import stats
import statsmodels.formula.api as smf
from urllib3.exceptions import InsecureRequestWarning

# pyalex — replaces all raw OpenAlex HTTP calls
import pyalex
from pyalex import Authors, Works

warnings.simplefilter('ignore', InsecureRequestWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- CAPES API (unchanged) ---
CAPES_BASE_URL = "https://apigw-proxy.capes.gov.br/observatorio/data/observatorio/producao"
ID_PROGRAMA = ID_programa_capes
CAPES_PAGE_SIZE = 100

# --- pyalex global config (replaces manual session headers) ---
pyalex.config.email = "gabrielc.pule@gmail.com"   # polite-pool
pyalex.config.api_key = oa_api_key                 # premium pool if available
pyalex.config.max_retries = 5
pyalex.config.retry_backoff_factor = 0.4
pyalex.config.retry_http_codes = [429, 500, 503]

# Fields we actually need from a Work (select= dramatically reduces payload)
# authorships already carries the nested institutions[] array per author
WORK_SELECT = (
    "id,display_name,doi,publication_year,cited_by_count,fwci,"
    "authorships,topics"
)

# Fields we actually need from an Author record
# affiliations carries institution + years history
AUTHOR_SELECT = (
    "id,display_name,works_count,cited_by_count,summary_stats,topics,affiliations"
)

print("✅ Dependências prontas e ambiente configurado.")


### 2. Data Extraction & Author Resolution
- **CAPES**: Downloads all production records for the specified `ID_PROGRAMA` (unchanged).
- **OpenAlex – author search (pyalex)**: Each name is resolved with `Authors().search(name).select(["id"]).get(per_page=1)`.  
  - `select(["id"])` means the server only returns the author ID — nothing else travels over the wire.  
  - pyalex reuses a single HTTP session and retries on 429/5xx automatically.

In [ ]:
# ── Extração de Dados da CAPES e Resolução de Autores no OpenAlex ────────────
import pickle
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

def baixar_dados_producao(id_programa):
    print(f"Verificando id-programa da CAPES: {id_programa}...")
    try:
        url_teste = f"{CAPES_BASE_URL}?query=id-programa:({id_programa})&page=0&size=1"
        response = requests.get(url_teste, verify=False)
        if not response.json().get('content', response.json()):
            return None
    except Exception as e:
        print(f"Erro CAPES: {e}")
        return None

    todos_registros = []
    page = 0
    while True:
        url_paginada = f"{CAPES_BASE_URL}?query=id-programa:({id_programa})&page={page}&size={CAPES_PAGE_SIZE}"
        try:
            resp = requests.get(url_paginada, verify=False)
            if resp.status_code != 200: break
            registros_pagina = resp.json().get('content', resp.json())
            if not registros_pagina: break
            todos_registros.extend(registros_pagina)
            if len(registros_pagina) < CAPES_PAGE_SIZE: break
            page += 1
            time.sleep(0.2)
        except Exception:
            break

    if todos_registros:
        df = pd.json_normalize(todos_registros)
        print(f"Download concluido! Producoes CAPES encontradas: {len(todos_registros)}")
        return df
    return None

# --- CAPES checkpoint (pickle preserves nested autores column) ---
CKPT_CAPES = CHECKPOINT_DIR / f"capes_{ID_PROGRAMA}.pkl"
if CKPT_CAPES.exists():
    with open(CKPT_CAPES, "rb") as _f:
        df_resultado = pickle.load(_f)
    print(f"✅ CAPES checkpoint carregado: {len(df_resultado)} producoes")
else:
    df_resultado = baixar_dados_producao(ID_PROGRAMA)
    with open(CKPT_CAPES, "wb") as _f:
        pickle.dump(df_resultado, _f)
    print(f"Checkpoint CAPES salvo: {CKPT_CAPES}")

# Extrair autores únicos (operação rápida, sempre derivada de df_resultado)
df_autores_exploded = df_resultado.explode('autores')
df_autores_norm = pd.json_normalize(df_autores_exploded['autores'].dropna())
pesquisadores_unicos = df_autores_norm['nomePessoa'].unique()
df_pesquisadores = (pd.DataFrame(pesquisadores_unicos, columns=['nomePessoa'])
                    .sort_values('nomePessoa').reset_index(drop=True))
print(f"Pesquisadores unicos CAPES encontrados: {len(df_pesquisadores)}")

# --- OpenAlex author ID checkpoint ---
CKPT_FILTERED_AUTHORS = CHECKPOINT_DIR / f"filtered_authors_{ID_PROGRAMA}.csv"
if CKPT_FILTERED_AUTHORS.exists():
    df_filtered_authors = pd.read_csv(CKPT_FILTERED_AUTHORS)
    before = len(df_filtered_authors)
    df_filtered_authors = (df_filtered_authors
                           .drop_duplicates(subset=['open_alex_id'])
                           .reset_index(drop=True))
    if before > len(df_filtered_authors):
        print(f"  {before - len(df_filtered_authors)} IDs duplicados removidos do checkpoint")
    print(f"✅ Author ID checkpoint carregado: {len(df_filtered_authors)} IDs validos")
else:
    # Parallel resolution: each name gets one OpenAlex Authors().search() call.
    # ThreadPoolExecutor is safe here because each call creates its own HTTP request;
    # pyalex's retry/back-off fires per-thread on 429s.
    # MAX_WORKERS=8 stays well within the polite-pool rate limit (~10 req/s).
    MAX_WORKERS = 8
    names = df_pesquisadores['nomePessoa'].tolist()
    total  = len(names)

    def _resolve(name):
        try:
            res = (Authors()
                   .search(name)
                   .select(["id"])
                   .get(per_page=1))
            return name, (res[0]["id"] if res else None)
        except Exception:
            return name, None

    print(f"Iniciando busca paralela de {total} autores no OpenAlex "
          f"(workers={MAX_WORKERS})...")
    ids_map = {}
    done = 0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(_resolve, n): n for n in names}
        for future in as_completed(futures):
            name, oa_id = future.result()
            ids_map[name] = oa_id
            done += 1
            if done % 50 == 0 or done == total:
                found = sum(1 for v in ids_map.values() if v is not None)
                print(f"  {done}/{total} processados — {found} IDs encontrados ate agora")

    ids = [ids_map[n] for n in names]
    df_pesquisadores['open_alex_id'] = ids
    df_filtered_authors = (df_pesquisadores
                           .dropna(subset=['open_alex_id'])
                           .drop_duplicates(subset=['open_alex_id'])
                           .reset_index(drop=True))
    df_filtered_authors.to_csv(CKPT_FILTERED_AUTHORS, index=False)
    print(f"Checkpoint de IDs salvo: {CKPT_FILTERED_AUTHORS}")

print(f"✅ Encontrados {len(df_filtered_authors)} IDs validos do OpenAlex.")


### 3. Comprehensive Works Retrieval
This cell fetches full metadata for every paper by the identified researchers.

**Efficiency changes (pyalex):**
| | Before | After |
|---|---|---|
| Works batch size | 20 IDs / request | **50 IDs / request** |
| Works `per_page` | 100 | **200** (max) — ~half the pagination round-trips |
| Response payload | full Work object | **`select(WORK_SELECT)`** — only the 8 fields we use |
| Author detail batch | 20 | **50** (OpenAlex OR-filter limit) |
| Author payload | full Author object | **`select(AUTHOR_SELECT)`** — only 5 fields |
| Rate-limit handling | manual `time.sleep(0.2)` | **pyalex retry/back-off** |

In [ ]:
# ── Busca Compreensiva de Obras e Detalhamento de Autores ────────────────────
CKPT_WORKS          = CHECKPOINT_DIR / f"works_{ID_PROGRAMA}.csv"
CKPT_RELS           = CHECKPOINT_DIR / f"rels_{ID_PROGRAMA}.csv"
CKPT_AUTHOR_DETAILS = CHECKPOINT_DIR / f"author_details_{ID_PROGRAMA}.csv"

# Schemas — used to invalidate stale checkpoints when columns change
_RELS_REQUIRED_COLS    = {"institution_id", "institution_display_name"}
_AUTHORS_REQUIRED_COLS = {"last_institution_id", "last_institution_display_name",
                          "last_affiliation_year", "affiliations_json", "i10_index"}

# --- Works + Relations checkpoint ---
_rels_stale = (CKPT_RELS.exists()
               and bool(_RELS_REQUIRED_COLS - set(pd.read_csv(CKPT_RELS, nrows=0).columns)))
if _rels_stale:
    print(f"⚠️  Checkpoint de relações desatualizado (faltam colunas de instituição). Refazendo fetch…")

if CKPT_WORKS.exists() and CKPT_RELS.exists() and not _rels_stale:
    df_works = pd.read_csv(CKPT_WORKS)
    df_rels  = pd.read_csv(CKPT_RELS)
    df_rels  = df_rels.dropna(subset=["work_id", "publication_year"])
    df_rels["publication_year"] = df_rels["publication_year"].astype(int)
    print(f"✅ Works checkpoint carregado: {len(df_works)} obras, {len(df_rels)} relações")
else:
    author_ids_clean = [url.split('/')[-1] for url in df_filtered_authors['open_alex_id']]
    all_works_data = []
    AUTHOR_BATCH = 50

    print("⌛ Buscando obras detalhadas no OpenAlex via pyalex (cursor, per_page=200)…")
    for i in range(0, len(author_ids_clean), AUTHOR_BATCH):
        batch = author_ids_clean[i:i + AUTHOR_BATCH]
        author_filter = "|".join(batch)
        for page in (Works()
                     .filter(author={"id": author_filter},
                             has_doi=True,
                             is_retracted=False,
                             publication_year=">1990")  # limitar a obras recentes para evitar excesso de dados
                     .select(WORK_SELECT)
                     .paginate(method="cursor", per_page=200)):
            all_works_data.extend(page)

    unique_works_dict = {w['id']: w for w in all_works_data}
    all_unique_works = list(unique_works_dict.values())
    print(f"✅ Total de obras únicas recuperadas: {len(all_unique_works)}")

    enhanced_works_rows, author_mapping_rows = [], []
    for work in all_unique_works:
        work_id    = work.get('id', '')
        work_title = work.get('display_name', '')
        doi_full   = work.get('doi', '')

        authorships      = work.get('authorships', [])
        author_ids_str   = "|".join([str(a['author']['id'])           for a in authorships if 'author' in a and a['author'].get('id')])
        author_names_str = "|".join([str(a['author']['display_name']) for a in authorships if 'author' in a and a['author'].get('display_name')])

        enhanced_works_rows.append({
            'id': work_id, 'display_name': work_title, 'doi': doi_full,
            'publication_year': work.get('publication_year'),
            'cited_by_count': work.get('cited_by_count'), 'fwci': work.get('fwci'),
            'authors_ids': author_ids_str, 'authors_names': author_names_str
        })

        topics = work.get('topics', [])
        t1 = topics[0] if len(topics) > 0 else {}
        t2 = topics[1] if len(topics) > 1 else {}
        t3 = topics[2] if len(topics) > 2 else {}

        for auth in authorships:
            author_info = auth.get('author', {})
            # Primary institution = first entry in this authorship's institutions[]
            insts = auth.get('institutions') or []
            primary_inst = insts[0] if insts else {}

            author_mapping_rows.append({
                'work_id': work_id, 'work_title': work_title,
                'author_id': author_info.get('id'), 'author_name': author_info.get('display_name'),
                'author_position': auth.get('author_position'), 'author_orcid': author_info.get('orcid'),
                'institution_id':            primary_inst.get('id'),
                'institution_display_name':  primary_inst.get('display_name'),
                'publication_year': work.get('publication_year'), 'fwci': work.get('fwci'),
                'topic_display_name_1': t1.get('display_name'), 'topic_score_1': t1.get('score'),
                'topic_display_name_2': t2.get('display_name'), 'topic_score_2': t2.get('score'),
                'topic_display_name_3': t3.get('display_name'), 'topic_score_3': t3.get('score'),
                'cited_by_count': work.get('cited_by_count')
            })

    df_works = pd.DataFrame(enhanced_works_rows)
    df_rels  = pd.DataFrame(author_mapping_rows)
    df_rels  = df_rels.dropna(subset=["work_id", "publication_year"])
    df_rels["publication_year"] = df_rels["publication_year"].astype(int)

    df_works.to_csv(CKPT_WORKS, index=False)
    df_rels.to_csv(CKPT_RELS, index=False)
    print(f"💾 Checkpoints de obras e relações salvos")

# Marcar autores que pertencem ao programa CAPES (id_programa)
programa_ids = set(df_filtered_authors['open_alex_id'].dropna())
df_rels['is_programa']  = df_rels['author_id'].isin(programa_ids)
df_works['has_programa_author'] = df_works['authors_ids'].fillna('').apply(
    lambda s: any(aid in programa_ids for aid in s.split('|') if aid)
)
print(f"   └─ {df_rels['is_programa'].sum()} relações de autores do programa "
      f"({df_rels['author_id'].isin(programa_ids).sum()} de {len(df_rels)} totais)")

# --- Author details checkpoint ---
_authors_stale = (CKPT_AUTHOR_DETAILS.exists()
                  and bool(_AUTHORS_REQUIRED_COLS - set(pd.read_csv(CKPT_AUTHOR_DETAILS, nrows=0).columns)))
if _authors_stale:
    print(f"⚠️  Checkpoint de autores desatualizado (faltam colunas de afiliação). Refazendo fetch…")

if CKPT_AUTHOR_DETAILS.exists() and not _authors_stale:
    df_authors = pd.read_csv(CKPT_AUTHOR_DETAILS)
    print(f"✅ Author details checkpoint carregado: {len(df_authors)} autores")
else:
    print("⌛ Buscando info detalhada para co-autores via pyalex…")
    all_unique_author_ids = [aid.split('/')[-1]
                             for aid in df_rels['author_id'].dropna().unique()]
    comprehensive_authors = []
    AUTHOR_DETAIL_BATCH = 50

    for i in range(0, len(all_unique_author_ids), AUTHOR_DETAIL_BATCH):
        batch = all_unique_author_ids[i:i + AUTHOR_DETAIL_BATCH]
        try:
            results = (Authors()
                       .filter(openalex="|".join(batch))
                       .select(AUTHOR_SELECT)
                       .get(per_page=AUTHOR_DETAIL_BATCH))
            for author in results:
                topics = author.get('topics', [])
                affiliations = author.get('affiliations') or []

                # Flatten affiliations: [{institution:{id,display_name},years:[..]}]
                aff_flat = [{
                    'institution_id':   (a.get('institution') or {}).get('id'),
                    'institution_name': (a.get('institution') or {}).get('display_name'),
                    'years':            a.get('years') or []
                } for a in affiliations]

                # Most recent affiliation = highest year across all entries
                last_inst, last_year = {}, None
                if aff_flat:
                    ranked = sorted(
                        [a for a in aff_flat if a['years']],
                        key=lambda a: max(a['years']),
                        reverse=True
                    )
                    if ranked:
                        last_inst = {'id': ranked[0]['institution_id'],
                                     'display_name': ranked[0]['institution_name']}
                        last_year = max(ranked[0]['years'])

                comprehensive_authors.append({
                    'author_id':                    author.get('id'),
                    'display_name':                 author.get('display_name'),
                    'works_count':                  author.get('works_count'),
                    'cited_by_count':               author.get('cited_by_count'),
                    'h_index':                      (author.get('summary_stats') or {}).get('h_index'),
                    'i10_index':                    (author.get('summary_stats') or {}).get('i10_index'),
                    'topic_display_name_1':         topics[0]['display_name'] if topics else None,
                    'last_institution_id':          last_inst.get('id'),
                    'last_institution_display_name':last_inst.get('display_name'),
                    'last_affiliation_year':        last_year,
                    'affiliations_json':            json.dumps(aff_flat, ensure_ascii=False),
                })
        except Exception as e:
            print(f"⚠️  Lote {i//AUTHOR_DETAIL_BATCH + 1} falhou: {e}")

    df_authors = pd.DataFrame(comprehensive_authors)
    df_authors.to_csv(CKPT_AUTHOR_DETAILS, index=False)
    print(f"💾 Checkpoint de detalhes de autores salvo: {CKPT_AUTHOR_DETAILS}")

# Marcar autores do programa CAPES também em df_authors
df_authors['is_programa'] = df_authors['author_id'].isin(programa_ids)
print(f"   └─ {df_authors['is_programa'].sum()} autores do programa em df_authors")

print("✅ Datasets de Obras, Relações e Autores processados em memória.")


### 4. Thematic Vectorization & Timelines
Here, the research topics of every paper are converted into mathematical vectors.
- **Cosine Similarity**: Measures how similar two papers are.
- **Drift**: Calculates how much an author's 'research focus' shifts from one year to the next.

In [ ]:
import numpy as np
import pandas as pd
try:
    import pyopencl as cl
    import pyopencl.array as cla
    import pyopencl.clmath as clmath

    _ctx, _dev_name = None, None
    for _p in cl.get_platforms():
        _gpus = _p.get_devices(device_type=cl.device_type.GPU)
        if _gpus:
            _ctx = cl.Context(devices=[_gpus[0]])
            _dev_name = _gpus[0].name
            break
    if _ctx is None:
        raise RuntimeError("Nenhuma GPU OpenCL encontrada")
    _queue = cl.CommandQueue(_ctx)

    def _gpu_to_numpy(x):
        return x.get() if isinstance(x, cla.Array) else np.asarray(x)

    def _to_gpu(x):
        if isinstance(x, cla.Array): return x
        return cla.to_device(_queue, np.asarray(x, dtype=np.float32))

    class _CP:
        float32 = np.float32

        def array(self, x): return _to_gpu(x)

        def zeros(self, shape, dtype=None):
            return cla.zeros(_queue, shape, dtype=dtype or np.float32)

        def dot(self, u, v):
            return float(cla.dot(u, v).get())

        def clip(self, x, a_min, a_max=None):
            return _to_gpu(np.clip(_gpu_to_numpy(x), a_min, a_max).astype(np.float32))

        def log2(self, x):
            return clmath.log2(x)

        def sum(self, x, axis=None, keepdims=False):
            if axis is None:
                return float(cla.sum(x).get())
            return _to_gpu(
                _gpu_to_numpy(x).sum(axis=axis, keepdims=keepdims).astype(np.float32)
            )

        def mean(self, x, axis=None):
            if axis is None:
                return float(cla.sum(x).get()) / x.size
            return _to_gpu(
                _gpu_to_numpy(x).mean(axis=axis).astype(np.float32)
            )

        def where(self, cond, x, y):
            return _to_gpu(
                np.where(_gpu_to_numpy(cond).astype(bool),
                         _gpu_to_numpy(x), y).astype(np.float32)
            )

        class linalg:
            @staticmethod
            def norm(x):
                return float(cla.dot(x, x).get()) ** 0.5

    cp = _CP()
    HAS_GPU = True
    print(f"AMD GPU detectado via OpenCL: {_dev_name}")

except Exception as _e:
    import numpy as cp
    HAS_GPU = False
    _gpu_to_numpy = np.asarray
    print(f"GPU indisponivel. Usando CPU (NumPy). [{_e}]")

def cosine_sim_gpu(u, v):
    if u is None or v is None: return np.nan
    nu, nv = cp.linalg.norm(u), cp.linalg.norm(v)
    if nu == 0 or nv == 0: return np.nan
    return float(cp.dot(u, v) / (nu * nv))

def js_divergence_gpu(p, q, eps=1e-12):
    if p is None or q is None: return np.nan
    p, q = cp.clip(p, eps, None), cp.clip(q, eps, None)
    p, q = p / cp.sum(p), q / cp.sum(q)
    m = 0.5 * (p + q)
    def kl(a, b): return float(cp.sum(a * (cp.log2(a) - cp.log2(b))))
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)

def profile_up_to(tl_author: pd.DataFrame, year: int, include: bool = False):
    mask = (tl_author["publication_year"] < year if not include else tl_author["publication_year"] <= year)
    sub = tl_author.loc[mask, "topic_vec"]
    if sub.empty: return None
    vecs = cp.array(np.stack(sub.to_list()))
    return cp.mean(vecs, axis=0)

topic_cols = ["topic_display_name_1", "topic_display_name_2", "topic_display_name_3"]
score_cols = ["topic_score_1", "topic_score_2", "topic_score_3"]

# ── Checkpoints ────────────────────────────────────────────────────────────────────────────────
CKPT_VOCAB     = CHECKPOINT_DIR / f"vocab_{ID_PROGRAMA}.json"
CKPT_W_DICT    = CHECKPOINT_DIR / f"W_cpu_dict_{ID_PROGRAMA}.pkl"
CKPT_TIMELINES = CHECKPOINT_DIR / f"timelines_{ID_PROGRAMA}.csv"

if CKPT_VOCAB.exists() and CKPT_W_DICT.exists() and CKPT_TIMELINES.exists():
    vocab = json.loads(CKPT_VOCAB.read_text(encoding="utf-8"))
    idx   = {t: i for i, t in enumerate(vocab)}
    with open(CKPT_W_DICT, "rb") as _f:
        W_cpu_dict = pickle.load(_f)
    timelines = pd.read_csv(CKPT_TIMELINES)
    timelines["publication_year"] = timelines["publication_year"].astype(int)
    _zero = np.zeros(len(vocab), dtype=np.float32)
    timelines["topic_vec"] = timelines["work_id"].apply(
        lambda wid: W_cpu_dict.get(wid, _zero).copy())
    print(f"✅ Timelines checkpoint carregado: {len(timelines)} registros, vocab={len(vocab)}")
else:
    vocab = sorted(set().union(*[df_rels[c].dropna().unique() for c in topic_cols]))
    idx   = {t: i for i, t in enumerate(vocab)}

    work_rows = df_rels.drop_duplicates(subset="work_id").copy()
    V = np.zeros((len(work_rows), len(vocab)), dtype=np.float32)
    work_ids = work_rows["work_id"].tolist()

    print("⏳ Construindo matriz tematica no GPU...")
    for i, tc, sc in zip(range(3), topic_cols, score_cols):
        names = work_rows[tc].values
        scores = work_rows[sc].fillna(0).values.astype(np.float32)
        valid_mask = pd.notna(work_rows[tc])
        col_indices = np.array([idx.get(n, -1) for n in names])
        valid_indices = (col_indices != -1) & valid_mask
        if valid_indices.any():
            V[np.where(valid_indices)[0], col_indices[valid_indices]] += scores[valid_indices]

    row_sums = V.sum(axis=1, keepdims=True)
    V /= np.where(row_sums > 0, row_sums, 1.0)
    V = cp.array(V)
    W_cpu_dict = {wid: _gpu_to_numpy(V[i]) for i, wid in enumerate(work_ids)}

    timeline_rows = []
    for author_id, grp in df_rels.sort_values("publication_year").groupby("author_id"):
        grp = grp.sort_values("publication_year").reset_index(drop=True)
        cum = cp.zeros(len(vocab), dtype=cp.float32)
        n_prior = 0
        for r in grp.itertuples(index=False):
            wid = r.work_id
            if wid not in W_cpu_dict: continue
            v = cp.array(W_cpu_dict[wid])
            prior_mean = cum / n_prior if n_prior > 0 else None
            timeline_rows.append({
                "author_id": author_id, "work_id": wid, "publication_year": int(r.publication_year),
                "drift_cos_vs_prior": (1.0 - cosine_sim_gpu(v, prior_mean)) if prior_mean is not None else float("nan"),
                "drift_js_vs_prior": js_divergence_gpu(v, prior_mean) if prior_mean is not None else float("nan"),
                "topic_vec": W_cpu_dict[wid]
            })
            cum += v; n_prior += 1

    timelines = pd.DataFrame(timeline_rows)
    timelines["topic_vec"] = timelines["topic_vec"].apply(np.array)

    CKPT_VOCAB.write_text(json.dumps(vocab, ensure_ascii=False), encoding="utf-8")
    with open(CKPT_W_DICT, "wb") as _f:
        pickle.dump(W_cpu_dict, _f)
    timelines.drop(columns=["topic_vec"]).to_csv(CKPT_TIMELINES, index=False)
    print(f"Ὃe Timelines checkpoint salvo ({len(timelines)} registros, vocab={len(vocab)})")

print("✓ Matrizes tematicas e Timelines prontas.")


### 5. PPI (Publishing Performance Index)
Based on the methodology by **Simões & Crespo**, this cell calculates the PPI, which balances quantity (number of works) and quality (citations/FWCI).
- The **Publishing Performance Box (PPB)** categorizes authors into performance areas (A-H) relative to the program's average.

In [ ]:
# ── Calculo PPI e Visualizacao ──────────────────────────────────────────────────────────────────────────────────

import time
from scipy.sparse import csr_matrix

first_year_by_author = df_rels.groupby("author_id")["publication_year"].min().rename("first_year")
last_year_by_author  = df_rels.groupby("author_id")["publication_year"].max().rename("last_year")
n_works_in_sample    = df_rels.groupby("author_id").size().rename("n_works_in_sample")

def compute_ppi(df, np_col, nc_col, alpha=0.5):
    I, total_np, total_nc_sq = len(df), df[np_col].sum(), np.sqrt(df[nc_col]).sum()
    p = df[np_col] / total_np if total_np > 0 else 0.0
    c = np.sqrt(df[nc_col]) / total_nc_sq if total_nc_sq > 0 else 0.0
    return alpha * (p - 1/I) + (1 - alpha) * (c - 1/I)

def ppb_area_vec(p_eq, c_eq):
    pm = p_eq.values.astype(float)
    cm = c_eq.values.astype(float)
    return np.select(
        [pd.isna(p_eq) | pd.isna(c_eq),
         (pm>=0)&(cm>=0)&(pm>=cm),
         (pm>=0)&(cm>=0)&(pm<cm),
         (pm<0)&(cm>=0)&(np.abs(pm)<cm),
         (pm<0)&(cm>=0)&(np.abs(pm)>=cm),
         (pm<0)&(cm<0)&(np.abs(pm)>=np.abs(cm)),
         (pm<0)&(cm<0)&(np.abs(pm)<np.abs(cm)),
         (pm>=0)&(cm<0)&(pm>=np.abs(cm))],
        [None, "B", "A", "H", "D", "E", "F", "G"],
        default="C"
    )

AREA_COLORS = {"A":"#2196F3","B":"#4CAF50","C":"#8BC34A","D":"#03A9F4",
               "E":"#9E9E9E","F":"#BDBDBD","G":"#FF9800","H":"#FF5722"}

# ── Profiles checkpoint (pickle for fast load; CSV fallback for compatibility) ──
CKPT_PROFILES_PQ  = CHECKPOINT_DIR / f"profiles_{ID_PROGRAMA}.pkl"
CKPT_PROFILES_CSV = CHECKPOINT_DIR / f"profiles_{ID_PROGRAMA}.csv"
_PROF_REQUIRED = {"hhi_specialization", "shannon_entropy", "PPI_accumulated",
                  "ppb_area", "p_minus_eq", "c_minus_eq"}

def _ckpt_cols(path):
    if path.suffix == ".pkl":
        return set(pd.read_pickle(path).columns)
    return set(pd.read_csv(path, nrows=0).columns)

# Prefer parquet; fall back to CSV if it exists and parquet does not
if CKPT_PROFILES_PQ.exists():
    _ckpt_path, _stale = CKPT_PROFILES_PQ, bool(_PROF_REQUIRED - _ckpt_cols(CKPT_PROFILES_PQ))
elif CKPT_PROFILES_CSV.exists():
    _ckpt_path, _stale = CKPT_PROFILES_CSV, bool(_PROF_REQUIRED - _ckpt_cols(CKPT_PROFILES_CSV))
else:
    _ckpt_path, _stale = None, True

if _stale and _ckpt_path:
    print("Profiles checkpoint desatualizado. Recalculando...")

if _ckpt_path and not _stale:
    t0 = time.perf_counter()
    if _ckpt_path.suffix == ".pkl":
        profiles = pd.read_pickle(_ckpt_path)
    else:
        profiles = pd.read_csv(_ckpt_path)
    profiles["academic_age"] = profiles["academic_age"].astype("Int64")
    print(f"Profiles checkpoint carregado: {len(profiles):,} autores  ({time.perf_counter()-t0:.1f}s)")
else:
    # ── Vectorized HHI + Shannon entropy via sparse matmul ──────────────────
    print(f"Calculando HHI e entropia ({len(df_rels['author_id'].dropna().unique()):,} autores, metodo vetorizado)...")
    t0 = time.perf_counter()

    all_author_ids = df_rels["author_id"].dropna().unique()
    author_to_idx  = {aid: i for i, aid in enumerate(all_author_ids)}
    n_authors      = len(all_author_ids)

    work_ids_list = list(W_cpu_dict.keys())
    work_to_idx   = {wid: i for i, wid in enumerate(work_ids_list)}
    n_works       = len(work_ids_list)

    df_aw = (df_rels[["author_id","work_id"]].dropna().drop_duplicates())
    df_aw = df_aw[df_aw["work_id"].isin(work_to_idx)].copy()
    df_aw["a_idx"] = df_aw["author_id"].map(author_to_idx)
    df_aw["w_idx"] = df_aw["work_id"].map(work_to_idx)
    df_aw = df_aw.dropna(subset=["a_idx","w_idx"])
    a_idx = df_aw["a_idx"].astype(int).values
    w_idx = df_aw["w_idx"].astype(int).values

    print(f"  Index maps built  ({time.perf_counter()-t0:.1f}s)")
    t1 = time.perf_counter()

    A = csr_matrix((np.ones(len(a_idx), dtype=np.float32), (a_idx, w_idx)),
                   shape=(n_authors, n_works))
    V_works = np.stack([W_cpu_dict[wid] for wid in work_ids_list]).astype(np.float32)
    print(f"  V_works stacked: {V_works.shape}  ({time.perf_counter()-t1:.1f}s)")
    t1 = time.perf_counter()

    author_sums    = np.asarray(A.dot(V_works))
    count_per_auth = np.asarray(A.sum(axis=1)).ravel()
    print(f"  Sparse matmul done  ({time.perf_counter()-t1:.1f}s)")

    valid = count_per_auth > 0
    safe_count = np.where(valid, count_per_auth, 1.0)
    author_mean = author_sums / safe_count[:, np.newaxis]
    author_mean[~valid] = 0.0
    row_sums = author_mean.sum(axis=1, keepdims=True)
    safe_rows = np.where(row_sums > 0, row_sums, 1.0)
    author_mean = author_mean / safe_rows
    author_mean[~valid] = 0.0

    hhi_vals = (author_mean ** 2).sum(axis=1)
    eps = 1e-12
    H_vals = -(author_mean * np.log2(author_mean + eps)).sum(axis=1)

    spec_df = pd.DataFrame({
        "author_id":        all_author_ids,
        "hhi_specialization": np.where(valid, hhi_vals, float("nan")),
        "shannon_entropy":    np.where(valid, H_vals,   float("nan")),
    })
    print(f"  HHI/entropy for {int(valid.sum()):,} autores  ({time.perf_counter()-t0:.1f}s total)")

    profiles = (df_authors
                .merge(first_year_by_author, on="author_id", how="left")
                .merge(last_year_by_author,  on="author_id", how="left")
                .merge(n_works_in_sample,    on="author_id", how="left")
                .merge(spec_df,              on="author_id", how="left"))
    profiles["academic_age"] = (profiles["last_year"] - profiles["first_year"]).astype("Int64")
    profiles["PPI_accumulated"] = compute_ppi(profiles, "works_count", "cited_by_count")

    N_authors = len(profiles)
    p_eq = profiles["works_count"] / profiles["works_count"].sum() - 1/N_authors
    c_eq = (np.sqrt(profiles["cited_by_count"]) / np.sqrt(profiles["cited_by_count"]).sum()
            - 1/N_authors)
    profiles["p_minus_eq"], profiles["c_minus_eq"] = p_eq, c_eq
    # Vectorized ppb_area — one numpy pass instead of 235k Python calls
    profiles["ppb_area"] = ppb_area_vec(p_eq, c_eq)

    profiles.to_pickle(CKPT_PROFILES_PQ)
    print(f"Profiles checkpoint salvo: {CKPT_PROFILES_PQ}  ({len(profiles):,} autores)")

p_eq        = profiles["p_minus_eq"]
c_eq        = profiles["c_minus_eq"]
N_authors   = len(profiles)


# Visualizacao PPI
t0 = time.perf_counter()
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Publishing Performance Index (PPI)", fontsize=14, fontweight="bold")

# axes[0] — PPB scatter: one ax.scatter() call per area (≤8), not one per row
ax = axes[0]
for area, sub in profiles.dropna(subset=["p_minus_eq","c_minus_eq"]).groupby("ppb_area"):
    ax.scatter(sub["p_minus_eq"], sub["c_minus_eq"],
               color=AREA_COLORS.get(area, "#000"), alpha=0.7, s=40, zorder=3, label=area)
ax.axhline(0, color="black", linestyle="--")
ax.axvline(0, color="black", linestyle="--")
lim = max(abs(p_eq.max()), abs(c_eq.max())) * 1.1
ax.plot([-lim, lim], [lim, -lim], color="red", linestyle="-.", label="YY' (PPI=0)")
ax.set_xlabel("pi - 1/I"); ax.set_ylabel("ci - 1/I"); ax.set_title("Publishing Performance Box")

ax = axes[1]
ax.hist(profiles["PPI_accumulated"].dropna(), bins=20, color="#2E75B6", edgecolor="white")
ax.axvline(0, color="red", linestyle="--")
ax.set_title("Distribuicao do PPI Acumulado")

ax = axes[2]
area_vals = profiles["ppb_area"].fillna("?")
for area in area_vals.unique():
    sub = profiles[area_vals == area]
    ax.scatter(sub["academic_age"], sub["PPI_accumulated"],
               color=AREA_COLORS.get(area, "#000"), alpha=0.7, s=40, label=area)
ax.axhline(0, color="black", linestyle="--")
ax.set_title("PPI vs. Idade Academica por Area PPB")
ax.legend(ncol=4, fontsize=7)

plt.tight_layout()
plt.show()
print(f"✅ PPI calculado.  Visualizacao: {time.perf_counter()-t0:.1f}s")

# ── Figura 2: h-index e i10-index (raw values) ──────────────────────────────
if "h_index" in profiles.columns:
    fig2, axes2 = plt.subplots(2, 2, figsize=(13, 9))
    fig2.suptitle("h-index e i10-index — Valores Brutos", fontsize=14, fontweight="bold")

    for row_i, (metric_name, metric_col) in enumerate([("h-index","h_index"),("i10-index","i10_index")]):
        if metric_col not in profiles.columns: continue
        ax_hist, ax_scat = axes2[row_i]

        # Distribution (split by is_programa if available)
        if "is_programa" in profiles.columns:
            for grp_val, grp_df in profiles.groupby("is_programa"):
                lbl = "Programa CAPES" if grp_val else "Externos"
                col = "#1565C0" if grp_val else "#E65100"
                ax_hist.hist(grp_df[metric_col].dropna(), bins=30, alpha=0.65,
                             color=col, edgecolor="white", label=lbl)
        else:
            ax_hist.hist(profiles[metric_col].dropna(), bins=30, color="#2E75B6", edgecolor="white")
        ax_hist.set_title(f"Distribuição {metric_name}")
        ax_hist.set_xlabel(metric_name); ax_hist.legend(fontsize=8)

        # vs Academic Age
        if "is_programa" in profiles.columns:
            for grp_val, grp_df in profiles.dropna(subset=["academic_age",metric_col]).groupby("is_programa"):
                lbl = "Programa CAPES" if grp_val else "Externos"
                col = "#1565C0" if grp_val else "#E65100"
                ax_scat.scatter(grp_df["academic_age"], grp_df[metric_col],
                                label=lbl, color=col, alpha=0.5, s=20)
        else:
            ax_scat.scatter(profiles["academic_age"], profiles[metric_col],
                            alpha=0.5, s=20, color="#2E75B6")
        ax_scat.set_title(f"{metric_name} vs Idade Acadêmica")
        ax_scat.set_xlabel("Idade acadêmica (anos)"); ax_scat.set_ylabel(metric_name)
        ax_scat.legend(fontsize=8)

    plt.tight_layout()
    plt.show()
    print("✅ h-index e i10-index (brutos) visualizados.")


### 7. Test of Research Questions (RQ1–RQ7)

Tests each of the 7 RQs from the framework PDF using the dyad-level table built above.

| RQ | What it asks | Test |
|---|---|---|
| **RQ1** | Does first co-authorship cause a thematic jump? Bigger for **recurrent** vs **occasional** collaborators? | Welch's t (real vs placebo) + OLS on `is_recurrent` |
| **RQ2** | Does **initial thematic distance** predict convergence (inverted-U)? | OLS w/ quadratic `initial_thematic_dist` |
| **RQ3** | Do **higher-status (PPI)** collaborators pull the focal author's drift more? | OLS `delta_drift_3 ~ ppi_diff * collab_freq_log` |
| **RQ4** | Does **authorship position** (first/middle/last) moderate received influence? | OLS w/ categorical `focal_position_at_meeting` |
| **RQ5** | Are specialists (low **Shannon entropy**) more resistant to influence? | OLS `delta_drift_3 ~ focal_entropy * ppi_diff` |
| **RQ6** | Can **junior** collaborators (lower academic age) shift senior focals? | OLS on senior subset; coef of `aa_diff` |
| **RQ7** | Can **lower-PPI** collaborators shift high-PPI focals? | OLS on high-PPI subset; coef of `ppi_diff` |

All regressions use the existing `_ols()` helper (cluster-robust SEs by `focal`, NaN-aware). DV is `delta_drift_3 = drift_post_3 − drift_pre_3` (difference-in-differences). Multiple-testing correction is Bonferroni across the 7 tests.

In [ ]:
# ── Criação de Díades, Placebos e Testes OLS ─────────────────────────────────
focal_eligible = set(timelines.groupby("author_id").size()[lambda s: s >= 2].index)
tl_by_author = {aid: grp.sort_values("publication_year").reset_index(drop=True)
                for aid, grp in timelines.groupby("author_id")}
prof_by_author = profiles.set_index("author_id")

# Restrict focal to programa authors — this is scientifically correct (we study
# CAPES researcher drift) and avoids combinatorial explosion for large programs.
# Without this, program 1733 generates ~10M pairs from 235k co-authors.
_programa_ids_set = set(df_rels.loc[df_rels["is_programa"], "author_id"].unique())
print(f"Programa authors available as focal: {len(_programa_ids_set):,}")

import time as _time
_t0 = _time.perf_counter()
raw_pairs = []
for wid, grp in df_rels.groupby("work_id"):
    auts = grp["author_id"].unique().tolist()
    if len(auts) < 2: continue
    if len(auts) > 25: continue  # skip hyper-collaborative papers (>25 authors)
    yr = int(grp["publication_year"].iloc[0])
    # Only focal = programa author; collab = any co-author
    for a in auts:
        if a not in _programa_ids_set: continue
        for b in auts:
            if a != b:
                raw_pairs.append((a, b, wid, yr))
print(f"raw_pairs: {len(raw_pairs):,}  ({_time.perf_counter()-_t0:.1f}s)")

pairs_df = pd.DataFrame(raw_pairs, columns=["focal", "collab", "work_id", "year"])
dyad_summary = pairs_df.groupby(["focal", "collab"]).agg(
    meeting_year=("year", "min"), last_collab_year=("year", "max"),
    n_joint_works=("work_id", "nunique")).reset_index()

dyad_summary["collab_span"] = dyad_summary["last_collab_year"] - dyad_summary["meeting_year"] + 1
dyad_summary["collab_freq_log"] = np.log1p(dyad_summary["n_joint_works"])
print(f"dyad_summary: {len(dyad_summary):,} unique dyads  ({_time.perf_counter()-_t0:.1f}s)")
_t0 = _time.perf_counter()

def _mean_drift(sub, k): return float(sub["drift_cos_vs_prior"].head(k).mean()) if not sub.empty else np.nan

dyad_rows, placebo_rows = [], []
RNG = np.random.default_rng(2026)

print(f"Processando {len(dyad_summary):,} diades...")
_done = 0
for _, d in dyad_summary.iterrows():
    f, c, y = d["focal"], d["collab"], int(d["meeting_year"])
    if f not in focal_eligible or f not in tl_by_author or c not in tl_by_author: continue

    tlf, tlc = tl_by_author[f], tl_by_author[c]
    pre_f  = tlf[tlf["publication_year"] < y].sort_values("publication_year")
    post_f = tlf[tlf["publication_year"] > y].sort_values("publication_year")
    prof_f, prof_c = profile_up_to(tlf, y), profile_up_to(tlc, y)

    if pre_f.empty or post_f.empty or prof_f is None or prof_c is None: continue

    initial_dist = 1.0 - cosine_sim_gpu(cp.array(prof_f), cp.array(prof_c))

    dyad_rows.append({
        "focal": f, "collab": c, "meeting_year": y,
        "n_joint_works":  int(d["n_joint_works"]),
        "collab_freq_log": d["collab_freq_log"],
        "focal_age_before":  y - int(tlf["publication_year"].min()),
        "collab_age_before": y - int(tlc["publication_year"].min()),
        "initial_thematic_dist": initial_dist,
        "drift_pre_3":  _mean_drift(pre_f.iloc[::-1], 3),
        "drift_post_3": _mean_drift(post_f, 3)
    })
    _done += 1
    if _done % 500 == 0 or _done == len(dyad_summary):
        print(f"  {_done:,}/{len(dyad_summary):,} diades  ({_time.perf_counter()-_t0:.1f}s)")

    # Placebo: pick a random year that is not the real meeting year
    years_avail = sorted(tlf["publication_year"].unique())
    placebo_candidates = [yr for yr in years_avail[1:-1] if yr != y]
    if placebo_candidates:
        fake_y  = RNG.choice(placebo_candidates)
        p_pre_f  = tlf[tlf["publication_year"] < fake_y].sort_values("publication_year")
        p_post_f = tlf[tlf["publication_year"] > fake_y].sort_values("publication_year")
        if not p_pre_f.empty and not p_post_f.empty:
            placebo_rows.append({
                "focal": f,
                "drift_pre_3":  _mean_drift(p_pre_f.iloc[::-1], 3),
                "drift_post_3": _mean_drift(p_post_f, 3)
            })

dyads    = pd.DataFrame(dyad_rows)
placebos = pd.DataFrame(placebo_rows)

if not dyads.empty:    dyads["delta_drift_3"]    = dyads["drift_post_3"]    - dyads["drift_pre_3"]
if not placebos.empty: placebos["delta_drift_3"] = placebos["drift_post_3"] - placebos["drift_pre_3"]

# Modelagem OLS — drop rows with NaN in any model variable BEFORE fitting,
# so cluster groups align with the residual matrix.
def _ols(formula, data, group_col="focal"):
    """NaN-safe cluster-robust OLS.
    Uses regex to extract column names from formula so interaction terms
    (*, :) and patsy functions (I(), C()) are handled correctly.
    """
    import re as _re
    if len(data) < 10: return None
    lhs, rhs = formula.split("~", 1)
    dep = lhs.strip()
    # Extract word tokens from RHS; keep only those that are actual column names
    tokens = set(_re.findall(r"\b([a-zA-Z_]\w*)\b", rhs))
    used_cols = list({group_col, dep} | (tokens & set(data.columns)))
    clean = data.dropna(subset=used_cols).reset_index(drop=True)
    if len(clean) < 10: return None
    return (smf.ols(formula, data=clean)
               .fit(cov_type="cluster", cov_kwds={"groups": clean[group_col]}))

print("=" * 60 + "\nPLACEBO — Colaboração real vs. pseudo-encontro\n" + "=" * 60)
if not dyads.empty and not placebos.empty:
    t_stat, p_val = stats.ttest_ind(
        dyads["delta_drift_3"].dropna(), placebos["delta_drift_3"].dropna(), equal_var=False)
    print(f"Teste t: t={t_stat:.3f}, p={p_val:.4f} {'(Impacto Sig.!)' if p_val < 0.05 else ''}")

print("\n" + "=" * 60 + "\nRQ1 / RQ2 — Impacto Frequência e Diferença Temática\n" + "=" * 60)
m1 = _ols("delta_drift_3 ~ collab_freq_log + initial_thematic_dist + focal_age_before + collab_age_before", dyads)
if m1: print(m1.summary().tables[1])

print("\n🚀 Análise e extração completas!")


In [ ]:
# ── Teste das 7 Questões de Pesquisa (RQ1–RQ7) ───────────────────────────────
# Reutiliza dyads, placebos, profiles, df_rels e o helper _ols() já definidos.

# ─── 1. Engenharia de features no nível da díade ─────────────────────────────
ppi_lookup = profiles.set_index("author_id")["PPI_accumulated"]
aa_lookup  = profiles.set_index("author_id")["academic_age"].astype("float")
ent_lookup = profiles.set_index("author_id")["shannon_entropy"]

dyads["focal_ppi"]     = dyads["focal"].map(ppi_lookup)
dyads["collab_ppi"]    = dyads["collab"].map(ppi_lookup)
dyads["ppi_diff"]      = dyads["collab_ppi"] - dyads["focal_ppi"]   # >0 → collab é mais sênior em PPI
dyads["focal_aa"]      = dyads["focal"].map(aa_lookup)
dyads["collab_aa"]     = dyads["collab"].map(aa_lookup)
dyads["aa_diff"]       = dyads["collab_aa"] - dyads["focal_aa"]    # <0 → collab é mais jovem
dyads["focal_entropy"] = dyads["focal"].map(ent_lookup)
dyads["is_recurrent"]  = (dyads["n_joint_works"] > 1).astype(int)

# h-index and i10-index differentials (same sign convention as ppi_diff)
if "PPI_h" in profiles.columns:
    h_lookup   = profiles.set_index("author_id")["h_index"]
    i10_lookup = profiles.set_index("author_id")["i10_index"]
    dyads["focal_h"]    = dyads["focal"].map(h_lookup)
    dyads["collab_h"]   = dyads["collab"].map(h_lookup)
    dyads["h_diff"]     = dyads["collab_h"] - dyads["focal_h"]
    dyads["focal_i10"]  = dyads["focal"].map(i10_lookup)
    dyads["collab_i10"] = dyads["collab"].map(i10_lookup)
    dyads["i10_diff"]   = dyads["collab_i10"] - dyads["focal_i10"]

# Posição de autoria do focal no ano do encontro (RQ4)
position_at_meeting = (df_rels
    .groupby(["author_id", "publication_year"])["author_position"]
    .agg(lambda s: s.mode().iat[0] if not s.mode().empty else None)
    .rename("focal_position_at_meeting"))
dyads = dyads.merge(position_at_meeting,
                    left_on=["focal", "meeting_year"],
                    right_index=True, how="left")

print(f"Díades disponíveis: {len(dyads)}")
print(f"  └─ com is_recurrent=1: {int(dyads['is_recurrent'].sum())}")
print(f"  └─ com posição de autoria conhecida: {dyads['focal_position_at_meeting'].notna().sum()}")

# ─── 2. Definição dos 7 testes ───────────────────────────────────────────────
# Cada entrada: (RQ, fórmula, dataset, coef de interesse).
high_aa_q  = dyads["focal_aa"].quantile(0.66)
high_ppi_q = dyads["focal_ppi"].quantile(0.66)
senior_subset   = dyads[dyads["focal_aa"]  >= high_aa_q].copy()
hi_ppi_subset   = dyads[dyads["focal_ppi"] >= high_ppi_q].copy()

rq_specs = [
    ("RQ1", "delta_drift_3 ~ is_recurrent",
            dyads, "is_recurrent"),
    ("RQ2", "delta_drift_3 ~ initial_thematic_dist + I(initial_thematic_dist**2)",
            dyads, "I(initial_thematic_dist ** 2)"),
    ("RQ3", "delta_drift_3 ~ ppi_diff * collab_freq_log + focal_age_before",
            dyads, "ppi_diff"),
    ("RQ4", "delta_drift_3 ~ C(focal_position_at_meeting, Treatment('middle')) + ppi_diff",
            dyads, "C(focal_position_at_meeting, Treatment('middle'))[T.first]"),
    ("RQ5", "delta_drift_3 ~ focal_entropy * ppi_diff",
            dyads, "focal_entropy:ppi_diff"),
    ("RQ6", "delta_drift_3 ~ aa_diff",
            senior_subset, "aa_diff"),
    ("RQ7", "delta_drift_3 ~ ppi_diff",
            hi_ppi_subset, "ppi_diff"),
]

# h-index and i10-index variants of RQ3 and RQ7
if "h_diff" in dyads.columns:
    hi_h_q   = dyads["focal_h"].quantile(0.66)   # focal.h_index threshold
    hi_i10_q = dyads["focal_i10"].quantile(0.66) # focal.i10_index threshold
    rq_specs += [
        ("RQ3_h",   "delta_drift_3 ~ h_diff * collab_freq_log + focal_age_before",
                    dyads, "h_diff"),
        ("RQ3_i10", "delta_drift_3 ~ i10_diff * collab_freq_log + focal_age_before",
                    dyads, "i10_diff"),
        ("RQ7_h",   "delta_drift_3 ~ h_diff",
                    dyads[dyads["focal_h"]   >= hi_h_q].copy(),   "h_diff"),
        ("RQ7_i10", "delta_drift_3 ~ i10_diff",
                    dyads[dyads["focal_i10"] >= hi_i10_q].copy(), "i10_diff"),
    ]

# ─── 3. Executar e coletar resultados ────────────────────────────────────────
results_rows = []
fitted_models = {}
for rq, formula, data, focal_coef in rq_specs:
    model = _ols(formula, data, group_col="focal")
    if model is None:
        results_rows.append({
            "RQ": rq, "n": len(data.dropna(subset=["delta_drift_3", "focal"])),
            "coef_of_interest": focal_coef, "coef": np.nan,
            "std_err": np.nan, "t": np.nan, "p_raw": np.nan
        })
        continue
    fitted_models[rq] = model
    if focal_coef in model.params.index:
        coef    = model.params[focal_coef]
        se      = model.bse[focal_coef]
        tval    = model.tvalues[focal_coef]
        pval    = model.pvalues[focal_coef]
    else:
        # Fallback: take the largest |t| among non-intercept terms
        non_int = [k for k in model.params.index if k != "Intercept"]
        if non_int:
            k = max(non_int, key=lambda k: abs(model.tvalues[k]))
            coef, se, tval, pval = model.params[k], model.bse[k], model.tvalues[k], model.pvalues[k]
            focal_coef = k + "  (fallback)"
        else:
            coef = se = tval = pval = np.nan
    results_rows.append({
        "RQ": rq, "n": int(model.nobs),
        "coef_of_interest": focal_coef, "coef": coef,
        "std_err": se, "t": tval, "p_raw": pval
    })

results = pd.DataFrame(results_rows)

# Bonferroni
n_tests = results["p_raw"].notna().sum()
results["p_bonferroni"] = (results["p_raw"] * n_tests).clip(upper=1.0)
def _stars(p):
    if pd.isna(p): return ""
    if p < 0.01: return "**"
    if p < 0.05: return "*"
    return ""
results["sig_bonf"] = results["p_bonferroni"].apply(_stars)

# ─── 4. RQ1 placebo (real vs pseudo-encontros) — controle adicional ──────────
print("=" * 72 + "\nRQ1 — Placebo: drift real vs pseudo-encontro\n" + "=" * 72)
if not dyads.empty and not placebos.empty:
    t_stat, p_val = stats.ttest_ind(
        dyads["delta_drift_3"].dropna(),
        placebos["delta_drift_3"].dropna(),
        equal_var=False)
    print(f"Welch t = {t_stat:.3f},  p = {p_val:.4f}  "
          f"{'(efeito da colaboração real significativo)' if p_val < 0.05 else '(não distinguível do placebo)'}")
else:
    print("Sem placebos disponíveis.")

# ─── 5. Tabela resumo ────────────────────────────────────────────────────────
print("\n" + "=" * 72 + f"\nResumo dos {len(rq_specs)} RQs (cluster-robusto por focal)\n" + "=" * 72)
print(results.to_string(index=False,
    formatters={"coef": "{:.4f}".format, "std_err": "{:.4f}".format,
                "t": "{:.2f}".format, "p_raw": "{:.4f}".format,
                "p_bonferroni": "{:.4f}".format}))

print("\nLegenda:  ** p<0.01 (Bonferroni),  * p<0.05 (Bonferroni)")

# Avisos de amostra pequena
small = results[results["n"] < 30]
if not small.empty:
    print(f"\n⚠️  RQ(s) com n<30 (interpretar com cautela): {small['RQ'].tolist()}")

# ─── 6. Persistir tabela ─────────────────────────────────────────────────────
RQ_OUT = CHECKPOINT_DIR / f"rq_results_{ID_PROGRAMA}.csv"
results.to_csv(RQ_OUT, index=False)
print(f"\n💾 Resultados salvos em {RQ_OUT}")


### 💾 Data Export
Run the cell below to export all processed datasets to CSV and download them to your local machine.

In [ ]:
# ── Exportação: CSVs + Relatório PDF (dois grupos: Programa CAPES vs Externos) ─────────────────
#
# Viz escolhidas por data-to-viz.com para cada insight:
#   • PPI por grupo (1 num × 1 cat)              → histograma sobrepostos + boxplot
#   • PPB area por grupo (1 cat × 1 cat)         → barras agrupadas
#   • PPI × academic_age por grupo (2 num)       → scatter (duas séries)
#   • Drift ao longo do tempo por grupo          → boxplot por ano (duas séries)
#   • Real vs placebo por grupo                  → violin (hue=grupo)
#   • Recorrente vs ocasional por grupo          → boxplot (hue=grupo)
#   • Distância inicial × Δ drift por grupo      → scatter + ajuste quadrático (duas séries)
#   • Posição autoria × Δ drift por grupo        → boxplot (hue=grupo)
#   • Top 15 instituições: programa vs externos  → lollipop lado a lado
#   • Tabela de RQs                              → matplotlib table
#
# Saída: export/<ID_PROGRAMA>/  com CSVs + relatorio_<ID>.pdf

from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

EXPORT_DIR = Path("export") / ID_PROGRAMA
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# ── Rótulos e paleta dos dois grupos ──────────────────────────────────────────
GRP_LABEL   = {True: "Programa CAPES", False: "Externos"}
GRP_PALETTE = {True: "#1565C0", False: "#E65100"}   # azul escuro vs laranja escuro
GRP_ALPHA   = {True: 0.75, False: 0.55}

# ── Enriquecer DataFrames com is_programa onde necessário ─────────────────────
programa_author_ids = set(df_rels.loc[df_rels["is_programa"], "author_id"].unique())

# timelines: juntar is_programa pelo author_id
if "is_programa" not in timelines.columns:
    timelines["is_programa"] = timelines["author_id"].isin(programa_author_ids)

# profiles: já vem de df_authors que tem is_programa
if "is_programa" not in profiles.columns:
    profiles["is_programa"] = profiles["author_id"].isin(programa_author_ids)

# dyads: focal é programa?
if "focal_is_programa" not in dyads.columns:
    dyads["focal_is_programa"] = dyads["focal"].isin(programa_author_ids)

# ── 1. CSVs dos datasets-chave ──────────────────────────────────────────────
CSV_TARGETS = {
    "df_filtered_authors": df_filtered_authors,
    "df_works":            df_works,
    "df_rels":             df_rels,
    "df_authors":          df_authors,
    "profiles":            profiles,
    "dyads":               dyads,
    "placebos":            placebos,
    "rq_results":          results,
}
print("Exportando CSVs...")
for name, obj in CSV_TARGETS.items():
    if isinstance(obj, pd.DataFrame) and not obj.empty:
        path = EXPORT_DIR / f"{name}.csv"
        obj.to_csv(path, index=False, encoding="utf-8-sig")
        print(f"   ok {path.name}  ({len(obj)} linhas)")

# ── 2. Relatório PDF ────────────────────────────────────────────────────────
sns.set_style("whitegrid")
PDF_PATH = EXPORT_DIR / f"relatorio_{ID_PROGRAMA}.pdf"
print(f"\nConstruindo {PDF_PATH.name}...")

def _save(pdf, fig):
    fig.tight_layout()
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

def _legend_patches():
    import matplotlib.patches as mpatches
    return [mpatches.Patch(color=GRP_PALETTE[True],  label=GRP_LABEL[True]),
            mpatches.Patch(color=GRP_PALETTE[False], label=GRP_LABEL[False])]

with PdfPages(PDF_PATH) as pdf:

    # ── Capa ────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8.5, 11))
    ax.axis("off")
    ax.text(0.5, 0.78, "Relatório Bibliométrico", ha="center", fontsize=24, fontweight="bold")
    ax.text(0.5, 0.71, f"Programa CAPES {ID_PROGRAMA}", ha="center", fontsize=16, color="#555")
    n_prog = int(profiles["is_programa"].sum()) if "is_programa" in profiles.columns else "?"
    n_ext  = len(profiles) - n_prog if isinstance(n_prog, int) else "?"
    ax.text(0.5, 0.56,
            f"Pesquisadores do programa: {len(df_filtered_authors)}\n"
            f"Autores em profiles — programa: {n_prog}  |  externos: {n_ext}\n"
            f"Obras únicas: {len(df_works)}\n"
            f"Co-autores únicos: {len(df_authors)}\n"
            f"Díades de colaboração: {len(dyads)}",
            ha="center", fontsize=12)
    ax.text(0.5, 0.10, "Pipeline CAPES + OpenAlex (pyalex)",
            ha="center", fontsize=10, color="#888", style="italic")
    ax.legend(handles=_legend_patches(), loc="lower center", fontsize=11,
              bbox_to_anchor=(0.5, 0.18), frameon=False)
    _save(pdf, fig)

    # ── 1. PPI: histogramas sobrepostos + boxplot lado a lado ────────────────
    if "PPI_accumulated" in profiles.columns and profiles["PPI_accumulated"].notna().any():
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        fig.suptitle("PPI — Programa CAPES vs Externos", fontsize=14, fontweight="bold")
        for grp_val, grp_df in profiles.groupby("is_programa"):
            vals = grp_df["PPI_accumulated"].dropna()
            lbl  = GRP_LABEL[grp_val]
            col  = GRP_PALETTE[grp_val]
            axes[0].hist(vals, bins=25, alpha=GRP_ALPHA[grp_val],
                         color=col, edgecolor="white", label=lbl)
        axes[0].axvline(0, color="red", linestyle="--", linewidth=1)
        axes[0].set_title("Distribuição (histograma sobreposto)")
        axes[0].set_xlabel("PPI acumulado")
        axes[0].legend()

        grp_order = [True, False]
        grp_labels = [GRP_LABEL[g] for g in grp_order]
        data_bp = [profiles.loc[profiles["is_programa"] == g, "PPI_accumulated"].dropna()
                   for g in grp_order]
        bp = axes[1].boxplot(data_bp, labels=grp_labels, patch_artist=True)
        for patch, grp_val in zip(bp["boxes"], grp_order):
            patch.set_facecolor(GRP_PALETTE[grp_val])
            patch.set_alpha(0.75)
        axes[1].axhline(0, color="red", linestyle="--", linewidth=0.8)
        axes[1].set_title("Dispersão (boxplot)")
        axes[1].set_ylabel("PPI acumulado")
        _save(pdf, fig)

    # ── 2. PPB area: barras agrupadas ────────────────────────────────────────
    if "ppb_area" in profiles.columns and "is_programa" in profiles.columns:
        AREA_COLORS = {"A":"#2196F3","B":"#4CAF50","C":"#8BC34A","D":"#03A9F4",
                       "E":"#9E9E9E","F":"#BDBDBD","G":"#FF9800","H":"#FF5722"}
        areas = sorted(profiles["ppb_area"].dropna().unique())
        x = np.arange(len(areas))
        width = 0.35
        prog_counts = profiles[profiles["is_programa"]]["ppb_area"].value_counts()
        ext_counts  = profiles[~profiles["is_programa"]]["ppb_area"].value_counts()

        fig, ax = plt.subplots(figsize=(10, 4.5))
        bars1 = ax.bar(x - width/2, [prog_counts.get(a, 0) for a in areas],
                       width, label=GRP_LABEL[True],  color=GRP_PALETTE[True],  alpha=0.8)
        bars2 = ax.bar(x + width/2, [ext_counts.get(a, 0) for a in areas],
                       width, label=GRP_LABEL[False], color=GRP_PALETTE[False], alpha=0.8)
        ax.set_xticks(x); ax.set_xticklabels(areas)
        ax.set_title("Áreas PPB por grupo", fontweight="bold")
        ax.set_ylabel("Pesquisadores"); ax.legend()
        _save(pdf, fig)

    # ── 3. PPI × Academic Age (scatter, duas séries) ─────────────────────────
    if {"academic_age","PPI_accumulated","is_programa"}.issubset(profiles.columns):
        fig, ax = plt.subplots(figsize=(10, 5.5))
        for grp_val, grp_df in profiles.dropna(subset=["academic_age","PPI_accumulated"]).groupby("is_programa"):
            ax.scatter(grp_df["academic_age"], grp_df["PPI_accumulated"],
                       label=GRP_LABEL[grp_val], color=GRP_PALETTE[grp_val],
                       alpha=GRP_ALPHA[grp_val], s=40)
        ax.axhline(0, color="black", linestyle="--", linewidth=0.7)
        ax.set_xlabel("Idade acadêmica (anos)")
        ax.set_ylabel("PPI acumulado")
        ax.set_title("PPI vs. Idade Acadêmica — Programa CAPES vs Externos", fontweight="bold")
        ax.legend()
        _save(pdf, fig)

    # ── 4. Drift ao longo do tempo por grupo (boxplot por ano) ───────────────
    if "drift_cos_vs_prior" in timelines.columns and "is_programa" in timelines.columns:
        tl = timelines.dropna(subset=["drift_cos_vs_prior"])
        tl = tl[tl["publication_year"] >= tl["publication_year"].max() - 14].copy()
        tl["grupo"] = tl["is_programa"].map(GRP_LABEL)
        if not tl.empty:
            fig, ax = plt.subplots(figsize=(13, 5))
            sns.boxplot(data=tl, x="publication_year", y="drift_cos_vs_prior",
                        hue="grupo", palette={GRP_LABEL[True]: GRP_PALETTE[True],
                                              GRP_LABEL[False]: GRP_PALETTE[False]},
                        ax=ax, fliersize=1.5, linewidth=0.8)
            ax.set_title("Drift temático por ano e grupo", fontweight="bold")
            ax.set_xlabel("Ano"); ax.set_ylabel("Drift cos")
            plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
            ax.legend(title="Grupo", loc="upper left")
            _save(pdf, fig)

    # ── 5. RQ1: real vs placebo por grupo (violin) ───────────────────────────
    if not dyads.empty and not placebos.empty and "focal_is_programa" in dyads.columns:
        d_real = dyads[["delta_drift_3","focal_is_programa"]].dropna()
        d_real["tipo"] = "Real"
        d_real["grupo"] = d_real["focal_is_programa"].map(GRP_LABEL)
        d_plac = placebos[["delta_drift_3"]].dropna()
        d_plac["tipo"]  = "Placebo"
        d_plac["grupo"] = "Todos"
        compare = pd.concat([d_real, d_plac], ignore_index=True)
        compare["tipo_grupo"] = compare["tipo"] + " — " + compare["grupo"]
        order = (["Real — " + GRP_LABEL[True], "Real — " + GRP_LABEL[False], "Placebo — Todos"])
        palette = {"Real — " + GRP_LABEL[True]:  GRP_PALETTE[True],
                   "Real — " + GRP_LABEL[False]:  GRP_PALETTE[False],
                   "Placebo — Todos":              "#9E9E9E"}
        fig, ax = plt.subplots(figsize=(11, 5))
        sns.violinplot(data=compare, x="tipo_grupo", y="delta_drift_3",
                       order=[o for o in order if o in compare["tipo_grupo"].unique()],
                       palette=palette, ax=ax, inner="box", cut=0)
        ax.axhline(0, color="black", linestyle="--", linewidth=0.7)
        ax.set_title("RQ1 — Δ drift: real vs placebo por grupo", fontweight="bold")
        ax.set_xlabel(""); ax.set_ylabel("Δ drift (post − pre, k=3)")
        plt.setp(ax.get_xticklabels(), rotation=15, ha="right")
        _save(pdf, fig)

    # ── 6. RQ1b: recorrente vs ocasional por grupo (boxplot) ─────────────────
    if {"is_recurrent","delta_drift_3","focal_is_programa"}.issubset(dyads.columns):
        d = dyads.dropna(subset=["delta_drift_3"]).copy()
        d["tipo"]  = d["is_recurrent"].map({1: "Recorrente", 0: "Ocasional"})
        d["grupo"] = d["focal_is_programa"].map(GRP_LABEL)
        fig, ax = plt.subplots(figsize=(10, 5))
        sns.boxplot(data=d, x="tipo", y="delta_drift_3", hue="grupo",
                    palette={GRP_LABEL[True]: GRP_PALETTE[True],
                             GRP_LABEL[False]: GRP_PALETTE[False]},
                    order=["Ocasional", "Recorrente"], ax=ax)
        ax.axhline(0, color="black", linestyle="--", linewidth=0.7)
        ax.set_title("RQ1 — Δ drift por tipo de colaborador e grupo", fontweight="bold")
        ax.set_xlabel(""); ax.set_ylabel("Δ drift")
        ax.legend(title="Grupo")
        _save(pdf, fig)

    # ── 7. RQ2: distância inicial × Δ drift por grupo (scatter + ajuste) ─────
    if {"initial_thematic_dist","delta_drift_3","focal_is_programa"}.issubset(dyads.columns):
        d = dyads.dropna(subset=["initial_thematic_dist","delta_drift_3"])
        fig, ax = plt.subplots(figsize=(10, 5.5))
        for grp_val, grp_df in d.groupby("focal_is_programa"):
            col = GRP_PALETTE[grp_val]
            lbl = GRP_LABEL[grp_val]
            x   = grp_df["initial_thematic_dist"].values
            y   = grp_df["delta_drift_3"].values
            ax.scatter(x, y, alpha=0.3, s=14, color=col)
            if len(x) >= 10:
                coef = np.polyfit(x, y, 2)
                xs = np.linspace(x.min(), x.max(), 200)
                ax.plot(xs, np.polyval(coef, xs), color=col, linewidth=2,
                        label=f"{lbl}  ({coef[0]:.3f}·x²)")
        ax.axhline(0, color="black", linestyle="--", linewidth=0.7)
        ax.set_title("RQ2 — Distância temática inicial vs Δ drift por grupo", fontweight="bold")
        ax.set_xlabel("Distância temática inicial (1 − cos)")
        ax.set_ylabel("Δ drift")
        ax.legend(loc="best", fontsize=9)
        _save(pdf, fig)

    # ── 8. RQ4: posição de autoria × Δ drift por grupo ───────────────────────
    if {"focal_position_at_meeting","delta_drift_3","focal_is_programa"}.issubset(dyads.columns):
        d = dyads.dropna(subset=["delta_drift_3","focal_position_at_meeting"]).copy()
        d["grupo"] = d["focal_is_programa"].map(GRP_LABEL)
        order = [p for p in ["first","middle","last"] if p in d["focal_position_at_meeting"].unique()]
        if order:
            fig, ax = plt.subplots(figsize=(10, 5))
            sns.boxplot(data=d, x="focal_position_at_meeting", y="delta_drift_3",
                        hue="grupo",
                        palette={GRP_LABEL[True]: GRP_PALETTE[True],
                                 GRP_LABEL[False]: GRP_PALETTE[False]},
                        order=order, ax=ax)
            ax.axhline(0, color="black", linestyle="--", linewidth=0.7)
            ax.set_title("RQ4 — Δ drift por posição de autoria e grupo", fontweight="bold")
            ax.set_xlabel("Posição"); ax.set_ylabel("Δ drift")
            ax.legend(title="Grupo")
            _save(pdf, fig)

    # ── 9. Top 15 instituições — programa vs externos (lollipop) ─────────────
    if "institution_display_name" in df_rels.columns:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=False)
        fig.suptitle("Top 15 Instituições por Grupo", fontsize=14, fontweight="bold")
        for ax, grp_val, title in zip(
                axes,
                [True, False],
                [GRP_LABEL[True], GRP_LABEL[False]]):
            top = (df_rels[df_rels["is_programa"] == grp_val]["institution_display_name"]
                   .dropna().value_counts().head(15))
            if top.empty:
                ax.axis("off"); continue
            col = GRP_PALETTE[grp_val]
            ax.hlines(y=top.index, xmin=0, xmax=top.values, color=col, linewidth=2, alpha=0.7)
            ax.plot(top.values, top.index, "o", markersize=8, color=col)
            for v, lbl in zip(top.values, top.index):
                ax.text(v + max(top.values)*0.01, lbl, str(v), va="center", fontsize=7)
            ax.set_title(title, fontweight="bold")
            ax.set_xlabel("Ocorrências")
            ax.invert_yaxis()
        _save(pdf, fig)

    # ── H-INDEX and I10-INDEX pages (two-group) ──────────────────────────────
    # ── h-index and i10-index (raw values) ──────────────────────────────────
    for metric_name, metric_col in [("h-index","h_index"),("i10-index","i10_index")]:
        if metric_col not in profiles.columns: continue
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        fig.suptitle(f"{metric_name.upper()} — Programa CAPES vs Externos",
                     fontsize=14, fontweight="bold")
        for grp_val, grp_df in profiles.groupby("is_programa"):
            vals = grp_df[metric_col].dropna()
            axes[0].hist(vals, bins=30, alpha=GRP_ALPHA[grp_val],
                         color=GRP_PALETTE[grp_val], edgecolor="white",
                         label=GRP_LABEL[grp_val])
        axes[0].set_title(f"Distribuição {metric_name}")
        axes[0].set_xlabel(metric_name); axes[0].legend()
        for grp_val, grp_df in profiles.dropna(subset=["academic_age",metric_col]).groupby("is_programa"):
            axes[1].scatter(grp_df["academic_age"], grp_df[metric_col],
                            label=GRP_LABEL[grp_val], color=GRP_PALETTE[grp_val],
                            alpha=GRP_ALPHA[grp_val], s=20)
        axes[1].set_title(f"{metric_name} vs Idade Acadêmica")
        axes[1].set_xlabel("Idade acadêmica (anos)"); axes[1].set_ylabel(metric_name)
        axes[1].legend(fontsize=8)
        _save(pdf, fig)

    # ── 10. Resultados das 7 RQs (tabela) ────────────────────────────────────
    if not results.empty:
        fig, ax = plt.subplots(figsize=(11, 4 + 0.3 * len(results)))
        ax.axis("off")
        ax.set_title("Resultados das 7 Questões de Pesquisa", fontweight="bold",
                     fontsize=13, pad=14)
        tbl_df = results.copy()
        for col in ("coef","std_err","t","p_raw","p_bonferroni"):
            if col in tbl_df.columns:
                tbl_df[col] = tbl_df[col].apply(
                    lambda v: "—" if pd.isna(v) else (f"{v:.4f}" if col != "t" else f"{v:.2f}"))
        tbl = ax.table(cellText=tbl_df.values, colLabels=tbl_df.columns,
                       loc="center", cellLoc="center")
        tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1.0, 1.4)
        for j in range(len(tbl_df.columns)):
            tbl[0, j].set_facecolor("#37474F")
            tbl[0, j].set_text_props(color="white", fontweight="bold")
        _save(pdf, fig)

    info = pdf.infodict()
    info["Title"]   = f"Relatório Bibliométrico — Programa {ID_PROGRAMA}"
    info["Author"]  = "Pipeline CAPES + OpenAlex"
    info["Subject"] = "Análise de produção, PPI, drift temático e RQs (dois grupos)"

print(f"   ok {PDF_PATH.name} salvo")
print(f"\nExportacao completa em {EXPORT_DIR}/")


In [ ]:
# ── Relatório PDF — somente pesquisadores do Programa CAPES ──────────────────
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

EXPORT_DIR = Path("export") / ID_PROGRAMA
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_style("whitegrid")
PDF_CAPES = EXPORT_DIR / f"relatorio_{ID_PROGRAMA}_capes.pdf"
COL = "#1565C0"   # azul único para os autores do programa

# ── Subsets filtrados ao programa ─────────────────────────────────────────────
programa_author_ids = set(df_rels.loc[df_rels["is_programa"], "author_id"].unique())

prof_c  = profiles[profiles["is_programa"] == True].copy() if "is_programa" in profiles.columns           else profiles[profiles["author_id"].isin(programa_author_ids)].copy()

tl_c    = (timelines[timelines["author_id"].isin(programa_author_ids)].copy()
           if "is_programa" not in timelines.columns
           else timelines[timelines["is_programa"] == True].copy())

if "focal_is_programa" in dyads.columns:
    dy_c = dyads[dyads["focal_is_programa"] == True].copy()
elif "focal" in dyads.columns:
    dy_c = dyads[dyads["focal"].isin(programa_author_ids)].copy()
else:
    dy_c = dyads.copy()

AREA_COLORS = {"A":"#2196F3","B":"#4CAF50","C":"#8BC34A","D":"#03A9F4",
               "E":"#9E9E9E","F":"#BDBDBD","G":"#FF9800","H":"#FF5722"}

def _save(pdf, fig):
    fig.tight_layout()
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

print(f"Construindo {PDF_CAPES.name}  "
      f"({len(prof_c)} autores | {len(tl_c)} timeline rows | {len(dy_c)} diades)...")

with PdfPages(PDF_CAPES) as pdf:

    # ── Capa ──────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8.5, 11))
    ax.axis("off")
    ax.text(0.5, 0.78, "Relatório Bibliométrico", ha="center",
            fontsize=24, fontweight="bold")
    ax.text(0.5, 0.71, f"Programa CAPES {ID_PROGRAMA} — Pesquisadores do Programa",
            ha="center", fontsize=14, color="#555")
    ax.text(0.5, 0.56,
            f"Pesquisadores analisados: {len(prof_c)}\n"
            f"Registros de timeline: {len(tl_c)}\n"
            f"Díades (focal no programa): {len(dy_c)}",
            ha="center", fontsize=12)
    ax.text(0.5, 0.10, "Pipeline CAPES + OpenAlex (pyalex)",
            ha="center", fontsize=10, color="#888", style="italic")
    _save(pdf, fig)

    # ── 1. PPI: histograma + boxplot ─────────────────────────────────────────
    if "PPI_accumulated" in prof_c.columns and prof_c["PPI_accumulated"].notna().any():
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
        fig.suptitle("PPI — Pesquisadores do Programa CAPES", fontsize=14, fontweight="bold")
        vals = prof_c["PPI_accumulated"].dropna()
        axes[0].hist(vals, bins=25, color=COL, edgecolor="white", alpha=0.85)
        axes[0].axvline(0, color="red", linestyle="--", linewidth=1)
        axes[0].set_title("Distribuição (histograma)")
        axes[0].set_xlabel("PPI acumulado")
        axes[1].boxplot(vals, patch_artist=True,
                        boxprops=dict(facecolor=COL, alpha=0.75))
        axes[1].axhline(0, color="red", linestyle="--", linewidth=0.8)
        axes[1].set_title("Dispersão (boxplot)")
        axes[1].set_ylabel("PPI acumulado")
        axes[1].set_xticks([])
        _save(pdf, fig)

    # ── 2. PPB area: barras horizontais ordenadas ─────────────────────────────
    if "ppb_area" in prof_c.columns and prof_c["ppb_area"].notna().any():
        counts = prof_c["ppb_area"].value_counts().sort_index()
        fig, ax = plt.subplots(figsize=(9, 4.5))
        bars = ax.barh(counts.index, counts.values,
                       color=[AREA_COLORS.get(a, "#777") for a in counts.index])
        for bar, v in zip(bars, counts.values):
            ax.text(v + max(counts.values)*0.01, bar.get_y() + bar.get_height()/2,
                    str(v), va="center", fontsize=9)
        ax.set_title("Áreas PPB — Pesquisadores do Programa CAPES", fontweight="bold")
        ax.set_xlabel("Pesquisadores")
        ax.invert_yaxis()
        _save(pdf, fig)

    # ── 3. PPI × Academic Age (scatter) ──────────────────────────────────────
    if {"academic_age","PPI_accumulated","ppb_area"}.issubset(prof_c.columns):
        fig, ax = plt.subplots(figsize=(9, 5))
        for area, sub in prof_c.dropna(subset=["academic_age","PPI_accumulated"]).groupby("ppb_area"):
            ax.scatter(sub["academic_age"], sub["PPI_accumulated"],
                       label=area, alpha=0.75, s=45,
                       color=AREA_COLORS.get(area, "#777"))
        ax.axhline(0, color="black", linestyle="--", linewidth=0.7)
        ax.set_xlabel("Idade acadêmica (anos)")
        ax.set_ylabel("PPI acumulado")
        ax.set_title("PPI vs. Idade Acadêmica — Programa CAPES", fontweight="bold")
        ax.legend(title="PPB", ncol=4, fontsize=8, loc="lower right")
        _save(pdf, fig)

    # ── 4. Drift ao longo do tempo (boxplot por ano) ──────────────────────────
    if "drift_cos_vs_prior" in tl_c.columns and tl_c["drift_cos_vs_prior"].notna().any():
        tl_r = tl_c.dropna(subset=["drift_cos_vs_prior"])
        tl_r = tl_r[tl_r["publication_year"] >= tl_r["publication_year"].max() - 14]
        if not tl_r.empty:
            fig, ax = plt.subplots(figsize=(11, 4.5))
            sns.boxplot(data=tl_r, x="publication_year", y="drift_cos_vs_prior",
                        ax=ax, color=COL, fliersize=2)
            ax.set_title("Drift temático por ano — Programa CAPES", fontweight="bold")
            ax.set_xlabel("Ano de publicação")
            ax.set_ylabel("Drift cos")
            plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
            _save(pdf, fig)

    # ── 5. RQ1: real vs placebo (violin) ─────────────────────────────────────
    if not dy_c.empty and not placebos.empty and        dy_c["delta_drift_3"].notna().any() and placebos["delta_drift_3"].notna().any():
        compare = pd.concat([
            dy_c[["delta_drift_3"]].assign(grupo="Real (programa)"),
            placebos[["delta_drift_3"]].assign(grupo="Placebo")
        ], ignore_index=True).dropna()
        fig, ax = plt.subplots(figsize=(8, 4.5))
        sns.violinplot(data=compare, x="grupo", y="delta_drift_3",
                       palette={"Real (programa)": COL, "Placebo": "#9E9E9E"},
                       ax=ax, inner="box", cut=0)
        ax.axhline(0, color="black", linestyle="--", linewidth=0.7)
        ax.set_title("RQ1 — Δ drift: real vs placebo (programa CAPES)", fontweight="bold")
        ax.set_xlabel(""); ax.set_ylabel("Δ drift (post − pre, k=3)")
        _save(pdf, fig)

    # ── 6. RQ1b: recorrente vs ocasional (boxplot) ────────────────────────────
    if "is_recurrent" in dy_c.columns and dy_c["delta_drift_3"].notna().any():
        d = dy_c.dropna(subset=["delta_drift_3"]).copy()
        d["tipo"] = d["is_recurrent"].map({1: "Recorrente (>1 obra)", 0: "Ocasional (1 obra)"})
        fig, ax = plt.subplots(figsize=(8, 4.5))
        sns.boxplot(data=d, x="tipo", y="delta_drift_3", color=COL,
                    order=["Ocasional (1 obra)", "Recorrente (>1 obra)"], ax=ax)
        ax.axhline(0, color="black", linestyle="--", linewidth=0.7)
        ax.set_title("RQ1 — Δ drift por tipo de colaborador (programa CAPES)", fontweight="bold")
        ax.set_xlabel(""); ax.set_ylabel("Δ drift")
        _save(pdf, fig)

    # ── 7. RQ2: distância inicial × Δ drift (scatter + ajuste quadrático) ────
    if {"initial_thematic_dist","delta_drift_3"}.issubset(dy_c.columns):
        d = dy_c.dropna(subset=["initial_thematic_dist","delta_drift_3"])
        if len(d) >= 20:
            fig, ax = plt.subplots(figsize=(9, 5))
            ax.scatter(d["initial_thematic_dist"], d["delta_drift_3"],
                       alpha=0.35, s=16, color=COL)
            x = d["initial_thematic_dist"].values
            y = d["delta_drift_3"].values
            coef = np.polyfit(x, y, 2)
            xs = np.linspace(x.min(), x.max(), 200)
            ax.plot(xs, np.polyval(coef, xs), color="red", linewidth=2,
                    label=f"Ajuste: {coef[0]:.3f}x² + {coef[1]:.3f}x + {coef[2]:.3f}")
            ax.axhline(0, color="black", linestyle="--", linewidth=0.7)
            ax.set_title("RQ2 — Distância inicial vs Δ drift (programa CAPES)", fontweight="bold")
            ax.set_xlabel("Distância temática inicial (1 − cos)")
            ax.set_ylabel("Δ drift")
            ax.legend(fontsize=9)
            _save(pdf, fig)

    # ── 8. RQ4: posição de autoria × Δ drift (boxplot) ────────────────────────
    if "focal_position_at_meeting" in dy_c.columns:
        d = dy_c.dropna(subset=["delta_drift_3","focal_position_at_meeting"])
        order = [p for p in ["first","middle","last"] if p in d["focal_position_at_meeting"].unique()]
        if order:
            fig, ax = plt.subplots(figsize=(8, 4.5))
            sns.boxplot(data=d, x="focal_position_at_meeting", y="delta_drift_3",
                        order=order, color=COL, ax=ax)
            ax.axhline(0, color="black", linestyle="--", linewidth=0.7)
            ax.set_title("RQ4 — Δ drift por posição de autoria (programa CAPES)", fontweight="bold")
            ax.set_xlabel("Posição"); ax.set_ylabel("Δ drift")
            _save(pdf, fig)

    # ── h-index and i10-index (raw values, CAPES only) ───────────────────────
    for metric_name, metric_col in [("h-index","h_index"),("i10-index","i10_index")]:
        if metric_col not in prof_c.columns: continue
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
        fig.suptitle(f"{metric_name.upper()} — Pesquisadores do Programa CAPES",
                     fontsize=14, fontweight="bold")
        vals = prof_c[metric_col].dropna()
        axes[0].hist(vals, bins=30, color=COL, edgecolor="white", alpha=0.85)
        axes[0].set_title(f"Distribuição {metric_name}"); axes[0].set_xlabel(metric_name)
        axes[1].scatter(prof_c["academic_age"], prof_c[metric_col],
                        color=COL, alpha=0.5, s=20)
        axes[1].set_title(f"{metric_name} vs Idade Acadêmica")
        axes[1].set_xlabel("Idade acadêmica (anos)"); axes[1].set_ylabel(metric_name)
        _save(pdf, fig)

    # ── 9. Resultados das 7 RQs (tabela) ──────────────────────────────────────
    if not results.empty:
        fig, ax = plt.subplots(figsize=(11, 4 + 0.3 * len(results)))
        ax.axis("off")
        ax.set_title("Resultados das 7 Questões de Pesquisa", fontweight="bold",
                     fontsize=13, pad=14)
        tbl_df = results.copy()
        for col in ("coef","std_err","t","p_raw","p_bonferroni"):
            if col in tbl_df.columns:
                tbl_df[col] = tbl_df[col].apply(
                    lambda v: "—" if pd.isna(v) else (f"{v:.4f}" if col != "t" else f"{v:.2f}"))
        tbl = ax.table(cellText=tbl_df.values, colLabels=tbl_df.columns,
                       loc="center", cellLoc="center")
        tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1.0, 1.4)
        for j in range(len(tbl_df.columns)):
            tbl[0, j].set_facecolor("#1565C0")
            tbl[0, j].set_text_props(color="white", fontweight="bold")
        _save(pdf, fig)

    info = pdf.infodict()
    info["Title"]   = f"Relatório CAPES — Programa {ID_PROGRAMA}"
    info["Author"]  = "Pipeline CAPES + OpenAlex"
    info["Subject"] = "Análise restrita aos pesquisadores do programa"

print(f"   ok {PDF_CAPES.name} salvo em {EXPORT_DIR}/")


### 6. Collaboration Impact Analysis (Regression)
Finally, the notebook tests if collaborating with others affects a researcher's thematic drift. It creates 'Placebo' groups (pseudo-collaborations) to statistically verify if the observed changes in research topics are truly driven by the partnership or just by natural career progression.